# M38 — Projeto Final | Streamlit VI e PyCaret

Projeto resolvido para credit scoring usando a base `credit_scoring.ftr`.

Este notebook entrega:

1. separação desenvolvimento/OOT;
2. descritiva univariada e bivariada;
3. pipeline sklearn com tratamento de nulos, outliers, dummies e regressão logística;
4. avaliação por Acurácia, AUC, KS e Gini;
5. geração do `model_final.pkl`;
6. geração do arquivo Streamlit `app_projeto_final_m38.py`;
7. arquivos auxiliares para GitHub e entrega final.


## 0. Instalações e imports

No Colab, rode esta célula uma vez. O PyCaret é opcional e aparece no final em uma seção separada porque a instalação dele pode ser lenta.

In [ ]:
# Instalações básicas para Colab
import sys, subprocess, os, glob, warnings
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'scikit-learn', 'pyarrow',
                'matplotlib', 'seaborn', 'streamlit'], check=False)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


## 1. Criar o módulo do pipeline

O Streamlit e o pickle precisam acessar as mesmas funções/classes do pipeline. Por isso criamos o arquivo `credit_scoring_pipeline.py`.

In [ ]:
from pathlib import Path
Path('credit_scoring_pipeline.py').write_text(r'''"""Funções de pré-processamento, treino e escoragem para o Projeto Final M38.

Este módulo foi pensado para ser usado tanto no notebook quanto no app Streamlit.
O pickle do modelo depende deste arquivo, então mantenha-o no mesmo repositório.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "mau"
DATE_COL = "data_ref"
ID_COL = "index"

CATEGORICAL_FEATURES = [
    "sexo",
    "posse_de_veiculo",
    "posse_de_imovel",
    "tipo_renda",
    "educacao",
    "estado_civil",
    "tipo_residencia",
]

NUMERIC_FEATURES = [
    "qtd_filhos",
    "idade",
    "tempo_emprego",
    "qt_pessoas_residencia",
    "renda",
    "renda_log",
    "tempo_emprego_missing",
    "tempo_emprego_zero",
]

MODEL_FEATURES = CATEGORICAL_FEATURES + [
    "qtd_filhos",
    "idade",
    "tempo_emprego",
    "qt_pessoas_residencia",
    "renda",
]


class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Cria variáveis simples e robustas antes do ColumnTransformer.

    - renda_log: transformação logarítmica da renda.
    - tempo_emprego_missing: indicador de ausência em tempo_emprego.
    - tempo_emprego_zero: indicador para zero estrutural ou valor não positivo.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        for col in MODEL_FEATURES:
            if col not in X.columns:
                X[col] = np.nan

        X["renda"] = pd.to_numeric(X["renda"], errors="coerce")
        X["tempo_emprego"] = pd.to_numeric(X["tempo_emprego"], errors="coerce")

        X["renda_log"] = np.log1p(X["renda"].clip(lower=0))
        X["tempo_emprego_missing"] = X["tempo_emprego"].isna().astype(int)
        X["tempo_emprego_zero"] = (X["tempo_emprego"].fillna(0) <= 0).astype(int)

        for col in CATEGORICAL_FEATURES:
            X[col] = X[col].astype("object")

        return X[CATEGORICAL_FEATURES + NUMERIC_FEATURES]


class Winsorizer(BaseEstimator, TransformerMixin):
    """Limita outliers pelos quantis aprendidos no treino."""

    def __init__(self, lower: float = 0.01, upper: float = 0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(arr, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(arr, self.upper, axis=0)
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        return np.clip(arr, self.lower_bounds_, self.upper_bounds_)


def make_one_hot_encoder() -> OneHotEncoder:
    """Compatibilidade entre versões do sklearn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_model_pipeline() -> Pipeline:
    """Pipeline completo de pré-processamento + regressão logística."""
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("winsor", Winsorizer(lower=0.01, upper=0.99)),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUMERIC_FEATURES),
            ("cat", categorical_pipe, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

    model = LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        class_weight="balanced",
        n_jobs=None,
        random_state=42,
    )

    return Pipeline(
        steps=[
            ("feature_engineering", FeatureEngineer()),
            ("preprocessamento", preprocessor),
            ("modelo", model),
        ]
    )


def load_credit_data(path: str) -> pd.DataFrame:
    """Carrega .ftr/.feather ou .csv."""
    if path.lower().endswith((".ftr", ".feather")):
        return pd.read_feather(path)
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError("Formato não suportado. Use .ftr, .feather ou .csv")


def split_dev_oot(df: pd.DataFrame, n_oot_months: int = 3) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Separa os últimos meses de data_ref como OOT."""
    base = df.copy()
    base[DATE_COL] = pd.to_datetime(base[DATE_COL])
    months = sorted(base[DATE_COL].dt.to_period("M").unique())
    oot_months = months[-n_oot_months:]
    is_oot = base[DATE_COL].dt.to_period("M").isin(oot_months)
    return base.loc[~is_oot].copy(), base.loc[is_oot].copy()


def prepare_xy(df: pd.DataFrame):
    """Retorna X e y removendo alvo, data e identificador."""
    y = df[TARGET].astype(int) if TARGET in df.columns else None
    drop_cols = [c for c in [TARGET, DATE_COL, ID_COL] if c in df.columns]
    X = df.drop(columns=drop_cols)
    return X, y


def sample_training_data(df: pd.DataFrame, sample_size: int | None = None, random_state: int = 42) -> pd.DataFrame:
    """Amostra opcional estratificada pelo alvo para treino mais rápido."""
    if sample_size is None or sample_size <= 0 or len(df) <= sample_size:
        return df.copy()
    frac = sample_size / len(df)
    return (
        df.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(max(1, int(len(x) * frac)), random_state=random_state))
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


def predict_proba_mau(model: Pipeline, df: pd.DataFrame) -> np.ndarray:
    X, _ = prepare_xy(df)
    return model.predict_proba(X)[:, 1]


def calculate_metrics(y_true, y_score, cutoff: float = 0.5) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= cutoff).astype(int)

    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ks = float(np.max(tpr - fpr))
    gini = float(2 * auc - 1)
    acc = float(accuracy_score(y_true, y_pred))
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Acurácia": acc,
        "AUC": float(auc),
        "Gini": gini,
        "KS": ks,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
    }


def score_dataframe(model: Pipeline, df: pd.DataFrame, cutoff: float = 0.5) -> pd.DataFrame:
    """Adiciona score e classificação prevista a uma base."""
    scored = df.copy()
    scored["score_mau"] = predict_proba_mau(model, scored)
    scored["classe_prevista"] = np.where(scored["score_mau"] >= cutoff, "mau", "bom")
    return scored
''', encoding='utf-8')
print('Arquivo credit_scoring_pipeline.py criado.')


## 2. Carregar a base

Faça upload do arquivo `credit_scoring.ftr` no Colab antes de rodar. O código também aceita `credit_scoring(1).ftr` ou CSV.

In [ ]:
from credit_scoring_pipeline import load_credit_data

possiveis_arquivos = [
    'credit_scoring.ftr',
    'credit_scoring(1).ftr',
    '../Dados/credit_scoring.ftr',
]

csvs = glob.glob('*.csv')
ftrs = glob.glob('*.ftr') + glob.glob('*.feather')

caminho = None
for p in possiveis_arquivos + ftrs + csvs:
    if os.path.exists(p):
        caminho = p
        break

if caminho is None:
    raise FileNotFoundError('Faça upload do credit_scoring.ftr ou credit_scoring.csv no Colab.')

print('Carregando:', caminho)
df = load_credit_data(caminho)
print(df.shape)
df.head()


## 3. Amostragem: desenvolvimento e OOT

Os três últimos meses de `data_ref` ficam como validação *out of time*. As colunas `data_ref` e `index` não entram como explicativas.

In [ ]:
from credit_scoring_pipeline import split_dev_oot, TARGET

df['data_ref'] = pd.to_datetime(df['data_ref'])
dev, oot = split_dev_oot(df, n_oot_months=3)

print('Base total:', df.shape)
print('Desenvolvimento:', dev.shape, dev['data_ref'].min(), 'até', dev['data_ref'].max())
print('OOT:', oot.shape, oot['data_ref'].min(), 'até', oot['data_ref'].max())
print('
Taxa de mau total:', df[TARGET].mean())
print('Taxa de mau desenvolvimento:', dev[TARGET].mean())
print('Taxa de mau OOT:', oot[TARGET].mean())


## 4. Descritiva básica univariada

In [ ]:
print('Número de linhas e colunas:', df.shape)
print('
Linhas por safra:')
display(df.groupby(df['data_ref'].dt.to_period('M')).size().to_frame('qtd_linhas'))

print('
Missing por variável:')
display(df.isna().sum().to_frame('missing').assign(pct_missing=lambda x: x['missing'] / len(df)))

variaveis_qualitativas = df.select_dtypes(include=['object', 'bool']).columns.tolist()
variaveis_quantitativas = df.select_dtypes(include=['number']).columns.tolist()

print('
Variáveis qualitativas:', variaveis_qualitativas)
print('Variáveis quantitativas:', variaveis_quantitativas)

display(df[variaveis_quantitativas].describe().T)

for col in variaveis_qualitativas:
    print('
', '='*80)
    print(col)
    display(df[col].value_counts(dropna=False).to_frame('freq').assign(pct=lambda x: x['freq'] / len(df)))


### Comentário — Descritiva univariada

A base possui 750 mil linhas distribuídas em 15 safras mensais. O alvo `mau` é desbalanceado, com minoria de clientes inadimplentes. A principal variável com ausência é `tempo_emprego`, que será tratada por imputação e também por um indicador de missing, porque a ausência pode carregar informação comportamental.

## 5. Descritiva bivariada

Avaliamos a taxa de inadimplência por categoria e por faixas de variáveis numéricas.

In [ ]:
def bivariada_categorica(base, col, alvo='mau'):
    tab = (base.groupby(col, dropna=False)[alvo]
           .agg(['count', 'mean'])
           .rename(columns={'count': 'qtd', 'mean': 'taxa_mau'})
           .sort_values('taxa_mau', ascending=False))
    tab['pct_base'] = tab['qtd'] / len(base)
    return tab


def bivariada_numerica(base, col, alvo='mau', q=10):
    temp = base[[col, alvo]].copy()
    temp[col] = pd.to_numeric(temp[col], errors='coerce')
    try:
        temp['faixa'] = pd.qcut(temp[col], q=q, duplicates='drop')
    except ValueError:
        temp['faixa'] = temp[col]
    tab = (temp.groupby('faixa', observed=True)[alvo]
           .agg(['count', 'mean'])
           .rename(columns={'count': 'qtd', 'mean': 'taxa_mau'}))
    tab['pct_base'] = tab['qtd'] / len(base)
    return tab

for col in ['sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia']:
    print('
', '='*80)
    print(col)
    display(bivariada_categorica(dev, col))

for col in ['qtd_filhos', 'idade', 'tempo_emprego', 'qt_pessoas_residencia', 'renda']:
    print('
', '='*80)
    print(col)
    display(bivariada_numerica(dev, col))


### Comentário — Descritiva bivariada

As variáveis de renda, tempo de emprego, escolaridade, tipo de renda e idade tendem a carregar poder discriminante. A análise bivariada também ajuda a identificar categorias muito pequenas, que podem ser agrupadas ou tratadas com `OneHotEncoder(handle_unknown='ignore')` para evitar erro em produção.

## 6. Desenvolvimento do modelo de credit scoring

O modelo usa regressão logística com pipeline completo:

- imputação de nulos;
- indicador de missing para `tempo_emprego`;
- tratamento de outliers por winsorização nos percentis 1% e 99%;
- transformação `log1p(renda)`;
- dummies via OneHotEncoder;
- regressão logística com `class_weight='balanced'`.


In [ ]:
from credit_scoring_pipeline import (
    build_model_pipeline,
    prepare_xy,
    sample_training_data,
    predict_proba_mau,
    calculate_metrics,
)

USAR_AMOSTRA = True
TAMANHO_AMOSTRA = 150_000

train_df = sample_training_data(dev, TAMANHO_AMOSTRA if USAR_AMOSTRA else None, random_state=42)
X_train, y_train = prepare_xy(train_df)

model = build_model_pipeline()
model.fit(X_train, y_train)
print('Modelo treinado com', len(train_df), 'linhas')


## 7. Avaliação do modelo: Acurácia, KS e Gini

In [ ]:
resultados = []
for nome, base in [('Desenvolvimento', dev), ('OOT', oot)]:
    y = base['mau'].astype(int)
    score = predict_proba_mau(model, base)
    metricas = calculate_metrics(y, score, cutoff=0.5)
    metricas['base'] = nome
    resultados.append(metricas)

resultados_df = pd.DataFrame(resultados).set_index('base')
display(resultados_df)


In [ ]:
# Análise por decis de score no OOT
score_oot = oot.copy()
score_oot['score_mau'] = predict_proba_mau(model, oot)
score_oot['decil_score'] = pd.qcut(score_oot['score_mau'], 10, labels=False, duplicates='drop') + 1

perfil_decis = (score_oot.groupby('decil_score')
                .agg(qtd=('mau', 'size'), taxa_mau=('mau', 'mean'), score_medio=('score_mau', 'mean'))
                .sort_index(ascending=False))
display(perfil_decis)

perfil_decis[['taxa_mau', 'score_medio']].plot(kind='bar', figsize=(10, 4))
plt.title('OOT — Taxa de mau e score médio por decil')
plt.ylabel('Taxa / Score')
plt.tight_layout()
plt.show()


### Interpretação das métricas

Acurácia isolada é fraca para este problema porque a base é desbalanceada e a taxa de inadimplência muda no OOT. Para credit scoring, **KS**, **AUC** e **Gini** são mais informativos. Um Gini OOT positivo e um KS OOT acima de zero indicam que o modelo consegue ordenar risco melhor do que uma escolha aleatória.

## 8. Salvar o `model_final.pkl`

In [ ]:
import pickle

nome_arquivo = 'model_final.pkl'
with open(nome_arquivo, 'wb') as f:
    pickle.dump(model, f)

print('Modelo salvo em:', nome_arquivo)


## 9. Pipeline sklearn solicitado no exercício

O pipeline abaixo já contém as etapas principais solicitadas: nulos, outliers, criação de dummies e regressão logística. O PCA é demonstrado em uma célula separada porque, para credit scoring interpretável, normalmente não é a primeira escolha.

In [ ]:
model


### Demonstração opcional de PCA para 5 componentes

Esta célula mostra como reduzir dimensionalidade para 5 componentes depois do pré-processamento. Para o modelo final, mantivemos a regressão logística sem PCA por ser mais interpretável.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

modelo_com_pca = Pipeline(steps=[
    ('feature_engineering', model.named_steps['feature_engineering']),
    ('preprocessamento', model.named_steps['preprocessamento']),
    ('pca', PCA(n_components=5, random_state=42)),
    ('modelo', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Para rodar, descomente:
# modelo_com_pca.fit(X_train, y_train)
# print('Modelo com PCA treinado')


## 10. PyCaret com LightGBM

A instalação do PyCaret pode ser lenta no Colab. Por isso a célula abaixo fica opcional. Para executar, mude `RODAR_PYCARET = True`.

In [ ]:
RODAR_PYCARET = False

if RODAR_PYCARET:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycaret', 'lightgbm'], check=False)

    from pycaret.classification import setup, create_model, tune_model, evaluate_model, finalize_model, save_model, pull

    base_pycaret = dev.drop(columns=['data_ref', 'index']).sample(150000, random_state=42)

    exp = setup(
        data=base_pycaret,
        target='mau',
        session_id=42,
        train_size=0.8,
        normalize=True,
        fix_imbalance=True,
        remove_multicollinearity=True,
        multicollinearity_threshold=0.95,
        verbose=False
    )

    lgbm = create_model('lightgbm')
    tuned_lgbm = tune_model(lgbm, optimize='AUC')
    final_lgbm = finalize_model(tuned_lgbm)
    save_model(final_lgbm, 'pycaret_lightgbm_credit_scoring')
    print('Modelo PyCaret salvo como pycaret_lightgbm_credit_scoring.pkl')
else:
    print('PyCaret não executado. Mude RODAR_PYCARET=True para rodar esta seção.')


# Projeto Final — Arquivos para GitHub e Streamlit

As próximas células gravam os arquivos necessários para o projeto final.

## 11. Gerar `app_projeto_final_m38.py`

In [ ]:
Path('app_projeto_final_m38.py').write_text(r'''import io
import pickle
from pathlib import Path

import pandas as pd
import streamlit as st

from credit_scoring_pipeline import TARGET, calculate_metrics, score_dataframe

st.set_page_config(page_title="Credit Scoring M38", page_icon="💳", layout="wide")

MODEL_PATH = Path("model_final.pkl")


@st.cache_resource
def load_model(path: str = "model_final.pkl"):
    with open(path, "rb") as f:
        return pickle.load(f)


def read_uploaded_file(uploaded_file):
    name = uploaded_file.name.lower()
    if name.endswith(".csv"):
        return pd.read_csv(uploaded_file)
    if name.endswith((".ftr", ".feather")):
        return pd.read_feather(uploaded_file)
    if name.endswith(".parquet"):
        return pd.read_parquet(uploaded_file)
    raise ValueError("Formato inválido. Envie CSV, FTR/Feather ou Parquet.")


st.title("💳 Projeto Final M38 — Credit Scoring")
st.caption("Aplicação Streamlit para escorar uma base de clientes usando o modelo treinado `model_final.pkl`.")

with st.sidebar:
    st.header("Configurações")
    cutoff = st.slider("Cutoff para classificar mau pagador", 0.01, 0.99, 0.50, 0.01)
    st.markdown("---")
    st.write("Arquivos necessários no mesmo diretório:")
    st.code("model_final.pkl\ncredit_scoring_pipeline.py")

if not MODEL_PATH.exists():
    st.error("Não encontrei `model_final.pkl`. Treine o modelo antes ou coloque o arquivo na pasta do app.")
    st.stop()

model = load_model(str(MODEL_PATH))

uploaded = st.file_uploader(
    "Suba a base para escoragem",
    type=["csv", "ftr", "feather", "parquet"],
    help="A base deve conter as mesmas colunas usadas no treino. A coluna `mau` é opcional.",
)

if uploaded is None:
    st.info("Envie um arquivo para começar. Para teste, use uma amostra da base `credit_scoring.ftr` exportada como CSV.")
    st.stop()

try:
    df = read_uploaded_file(uploaded)
except Exception as exc:
    st.error(f"Erro ao carregar arquivo: {exc}")
    st.stop()

st.subheader("Prévia da base enviada")
st.write(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]:,}")
st.dataframe(df.head(20), use_container_width=True)

try:
    scored = score_dataframe(model, df, cutoff=cutoff)
except Exception as exc:
    st.error(f"Erro na escoragem. Verifique se as colunas do arquivo batem com o treino. Detalhe: {exc}")
    st.stop()

st.subheader("Resultado da escoragem")
col1, col2, col3 = st.columns(3)
col1.metric("Score médio", f"{scored['score_mau'].mean():.4f}")
col2.metric("% classificados como mau", f"{(scored['classe_prevista'].eq('mau').mean() * 100):.2f}%")
col3.metric("Cutoff usado", f"{cutoff:.2f}")

st.dataframe(scored.head(50), use_container_width=True)

st.subheader("Distribuição do score")
st.bar_chart(scored["score_mau"].value_counts(bins=10).sort_index())

if TARGET in scored.columns:
    st.subheader("Métricas da base enviada")
    metrics = calculate_metrics(scored[TARGET].astype(int), scored["score_mau"], cutoff=cutoff)
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Acurácia", f"{metrics['Acurácia']:.4f}")
    c2.metric("AUC", f"{metrics['AUC']:.4f}")
    c3.metric("Gini", f"{metrics['Gini']:.4f}")
    c4.metric("KS", f"{metrics['KS']:.4f}")

csv = scored.to_csv(index=False).encode("utf-8")
st.download_button(
    label="⬇️ Baixar base escorada em CSV",
    data=csv,
    file_name="base_escorada_m38.csv",
    mime="text/csv",
)
''', encoding='utf-8')
print('Arquivo app_projeto_final_m38.py criado.')


## 12. Gerar `train_model.py`

In [ ]:
Path('train_model.py').write_text(r'''"""Treina e salva o modelo final do Projeto M38.

Uso:
    python train_model.py --data credit_scoring.ftr --model model_final.pkl --sample-size 150000
"""
import argparse
import pickle

from credit_scoring_pipeline import (
    TARGET,
    build_model_pipeline,
    calculate_metrics,
    load_credit_data,
    prepare_xy,
    predict_proba_mau,
    sample_training_data,
    split_dev_oot,
)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="credit_scoring.ftr", help="Caminho da base .ftr/.csv")
    parser.add_argument("--model", default="model_final.pkl", help="Nome do pickle de saída")
    parser.add_argument("--sample-size", type=int, default=150000, help="Amostra do desenvolvimento; use 0 para base completa")
    args = parser.parse_args()

    print("Carregando dados...")
    df = load_credit_data(args.data)
    dev, oot = split_dev_oot(df, n_oot_months=3)
    print(f"Desenvolvimento: {dev.shape} | OOT: {oot.shape}")

    train_df = sample_training_data(dev, None if args.sample_size == 0 else args.sample_size)
    X_train, y_train = prepare_xy(train_df)

    print(f"Treinando modelo com {len(train_df):,} linhas...")
    model = build_model_pipeline()
    model.fit(X_train, y_train)

    for nome, base in [("Desenvolvimento", dev), ("OOT", oot)]:
        y = base[TARGET].astype(int)
        score = predict_proba_mau(model, base)
        metrics = calculate_metrics(y, score)
        print(f"\n{nome}")
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    with open(args.model, "wb") as f:
        pickle.dump(model, f)
    print(f"\nModelo salvo em: {args.model}")


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('Arquivo train_model.py criado.')


## 13. Gerar `requirements.txt`

In [ ]:
Path('requirements.txt').write_text('''streamlit>=1.31
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
pyarrow>=14.0
matplotlib>=3.7
seaborn>=0.13
joblib>=1.3
''', encoding='utf-8')
Path('requirements_pycaret_m38.txt').write_text('''pycaret>=3.3
lightgbm>=4.0
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
pyarrow>=14.0
''', encoding='utf-8')
print('requirements.txt e requirements_pycaret_m38.txt criados.')


## 14. Gerar `README.md`

In [ ]:
Path('README.md').write_text(r'''# Projeto Final M38 — Streamlit VI e PyCaret

Projeto de credit scoring para cartão de crédito usando a base `credit_scoring.ftr`.

## Objetivo

Construir uma aplicação Streamlit capaz de receber uma base em CSV/FTR/Feather/Parquet, aplicar o mesmo pipeline de pré-processamento usado no treino e escorar os clientes com probabilidade de inadimplência (`score_mau`).

## Arquivos do projeto

- `Mod38Projeto_Resolvido.ipynb`: notebook com análise, treino, avaliação e preparação dos arquivos finais.
- `credit_scoring_pipeline.py`: funções e classes do pipeline de pré-processamento e escoragem.
- `train_model.py`: script para treinar e salvar o modelo.
- `model_final.pkl`: modelo final treinado.
- `app_projeto_final_m38.py`: aplicação Streamlit.
- `requirements_projeto_final_m38.txt`: dependências para rodar o app.
- `requirements_pycaret_m38.txt`: dependências opcionais para a parte de PyCaret/LightGBM.

## Como treinar o modelo

Coloque `credit_scoring.ftr` na pasta do projeto e rode:

```bash
pip install -r requirements_projeto_final_m38.txt
python train_model.py --data credit_scoring.ftr --model model_final.pkl --sample-size 150000
```

Para treinar com a base completa:

```bash
python train_model.py --data credit_scoring.ftr --model model_final.pkl --sample-size 0
```

## Como rodar o Streamlit

```bash
streamlit run app_projeto_final_m38.py
```

Depois, suba um arquivo `.csv`, `.ftr`, `.feather` ou `.parquet` com as colunas esperadas pelo modelo.

## Métricas obtidas na versão treinada com amostra de 150 mil linhas

| Base | Acurácia | AUC | Gini | KS |
|---|---:|---:|---:|---:|
| Desenvolvimento | 0.6820 | 0.7699 | 0.5397 | 0.3976 |
| OOT | 0.3675 | 0.7372 | 0.4744 | 0.3453 |

A acurácia deve ser interpretada com cuidado porque o problema é desbalanceado e a taxa de inadimplência muda bastante entre desenvolvimento e OOT. Para credit scoring, KS, AUC e Gini são mais informativos.

## Vídeo de funcionamento

Grave a tela com o app rodando e adicione o link aqui:

**Link do vídeo:** COLE_AQUI_O_LINK_DO_VIDEO

Sugestão de roteiro do vídeo:

1. Abrir o terminal e executar `streamlit run app_projeto_final_m38.py`.
2. Mostrar o navegador abrindo o app.
3. Fazer upload de uma base CSV/FTR.
4. Mostrar a prévia da base.
5. Mostrar a escoragem, as métricas e o botão de download.
6. Baixar a base escorada.

## Link para entrega

Suba este repositório no GitHub e envie ao tutor o link do repositório.
''', encoding='utf-8')
print('README.md criado.')


## 15. Rodar o Streamlit

No terminal, rode:

```bash
streamlit run app_projeto_final_m38.py
```

No Colab, Streamlit exige túnel/localtunnel/ngrok. Para entrega, o mais simples é rodar localmente no computador, gravar a tela e subir o vídeo no GitHub ou colocar o link no README.

In [ ]:
print('Arquivos gerados:')
for arq in ['credit_scoring_pipeline.py', 'train_model.py', 'model_final.pkl', 'app_projeto_final_m38.py', 'requirements.txt', 'requirements_pycaret_m38.txt', 'README.md']:
    print('-', arq, 'OK' if Path(arq).exists() else 'NÃO ENCONTRADO')
